In [25]:
import wandb

api = wandb.Api()
# get all runs created today
runs = api.runs('learneddb/lcm', filters={'state': 'finished'})

In [34]:
from tqdm import tqdm
from datetime import datetime

runs_adjusted_ctr = 0
print(f'Found {len(runs)} runs')
for run in tqdm(runs):
    # get summary metrics
    summary = run.summary

    # retrieve and parse created_at
    created_at = run.created_at
    created_at = datetime.strptime(created_at, '%Y-%m-%dT%H:%M:%SZ')
    if created_at.date() < datetime.now().date():
        continue

    # compute new metrics
    selected_runtimes = []
    slowdowns = []
    try:
        for key in ['join_order_job_eval', 'join_order_job_light_eval', 'physop_eval', 'physop_eval_with_idxs',
                    'accesspath_eval']:
            selected_runtime = summary[f'{key}/total_selected_runtime']
            otimal_runtime = summary[f'{key}/total_optimal_runtime']

            selected_runtimes.append(selected_runtime)
            slowdowns.append(selected_runtime / otimal_runtime)
    except KeyError:
        print(f'Key not found: {key} in run {run.id}')
        continue

    runs_adjusted_ctr += 1

    run_id = run.id

    # continue run
    wandb_run = wandb.init(entity='learneddb', project='lcm', id=run_id, resume='must')
    wandb_run.summary.update({
        'qo_total/selected_runtime': sum(selected_runtimes),
        'qo_total/avg_slowdown': sum(slowdowns) / len(slowdowns)
    })

print(f'Adjusted {runs_adjusted_ctr} runs')



Found 131 runs


  0%|          | 0/131 [00:00<?, ?it/s]

access_path_selection.json/epoch_test_loss,137.21005
access_path_selection.json/epoch_test_mape,135.64479
access_path_selection.json/epoch_test_median_q_error_100,3473.88916
access_path_selection.json/epoch_test_median_q_error_50,3.1114
access_path_selection.json/epoch_test_median_q_error_95,93.29348
access_path_selection.json/epoch_test_mse,3.33091
access_path_selection.json/epoch_test_num_tuples,512
access_path_selection.json/epoch_test_spearman_rank,0.40035
access_path_selection.json/epoch_test_time,6.27863
access_path_selection.json/test_loss,24.17464
access_path_selection.json/test_mape,474.12903


 94%|█████████▍| 123/131 [00:04<00:00, 28.24it/s]/home/johannes/ldb/ldb_models/.venv/lib/python3.12/site-packages/wandb/sdk/wandb_run.py:2380: UserWarning: Run (gdirh3ou) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),


access_path_selection.json/test_mape,2096.85706
access_path_selection.json/test_median_q_error_100,54111.60912
access_path_selection.json/test_median_q_error_50,5.86568
access_path_selection.json/test_median_q_error_95,49.85711
access_path_selection.json/test_mse,3.25623
access_path_selection.json/test_no_test_points,506
access_path_selection.json/test_spearman_rank,0.89195
access_path_selection.json/test_test_time,0.00018
accesspath_eval/acc/cast_info.id,0.90909
accesspath_eval/acc/cast_info.movie_id,1
accesspath_eval/acc/cast_info.nr_order,0.45455


access_path_selection.json/test_mape,247.92896
access_path_selection.json/test_median_q_error_100,7515.71055
access_path_selection.json/test_median_q_error_50,2.01496
access_path_selection.json/test_median_q_error_95,585.88544
access_path_selection.json/test_mse,2.38479
access_path_selection.json/test_no_test_points,506
access_path_selection.json/test_spearman_rank,0.81431
access_path_selection.json/test_test_time,0.01164
accesspath_eval/acc/cast_info.id,0.18182
accesspath_eval/acc/cast_info.movie_id,0.54545
accesspath_eval/acc/cast_info.nr_order,0.63636


... (9 outputs left out)

 99%|█████████▉| 130/131 [00:36<00:00,  1.90it/s]/home/johannes/ldb/ldb_models/.venv/lib/python3.12/site-packages/wandb/sdk/wandb_run.py:2380: UserWarning: Run (gdirh3ou) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),


access_path_selection.json/test_mape,278.4926
access_path_selection.json/test_median_q_error_100,7691.82788
access_path_selection.json/test_median_q_error_50,2.03194
access_path_selection.json/test_median_q_error_95,15.13488
access_path_selection.json/test_mse,2.21253
access_path_selection.json/test_no_test_points,506
access_path_selection.json/test_spearman_rank,0.85655
access_path_selection.json/test_test_time,0.01108
accesspath_eval/acc/cast_info.id,0.45455
accesspath_eval/acc/cast_info.movie_id,0.81818
accesspath_eval/acc/cast_info.nr_order,0.63636


100%|██████████| 131/131 [00:39<00:00,  1.55it/s]/home/johannes/ldb/ldb_models/.venv/lib/python3.12/site-packages/wandb/sdk/wandb_run.py:2380: UserWarning: Run (gdirh3ou) is finished. The call to `_console_raw_callback` will be ignored. Please make sure that you are using an active run.
  lambda data: self._console_raw_callback("stderr", data),
100%|██████████| 131/131 [00:39<00:00,  3.28it/s]

Adjusted 9 runs


In [ ]:
keys = run.summary.keys()
keys = [k for k in keys if 'total_selected_runtime' in k]
print(keys)